# Marker-gene rank heatmap for prioritized IP/GV cNMF programs

This notebook creates a publication-quality marker-gene heatmap for P15, P9, P6, and P12.

The selected short labels are:

- **P15 — Homeostatic EC**
- **P9 — Angiogenic migration**
- **P6 — Inflammatory adhesion**
- **P12 — sGC-cGMP remodeling**

For every selected gene, the notebook calculates its rank in **all four programs** from the full K=30 cNMF spectra. Cells with rank above `MAX_RANK_TO_SHOW` are left blank. Fill is:

\[
1 / \log_2(\mathrm{rank}+1)
\]

so rank 1 has value 1 and lower-ranked genes fade toward white.

In [ ]:
import os

for variable in [
    "OMP_NUM_THREADS",
    "OPENBLAS_NUM_THREADS",
    "MKL_NUM_THREADS",
    "BLIS_NUM_THREADS",
    "NUMEXPR_NUM_THREADS",
]:
    os.environ.setdefault(variable, "1")


In [ ]:
from pathlib import Path
import math
import re

import matplotlib as mpl
import numpy as np
import pandas as pd
from IPython.display import display
from plotnine import (
    aes,
    element_blank,
    element_text,
    geom_text,
    geom_tile,
    geom_vline,
    ggplot,
    labs,
    scale_fill_gradient,
    theme,
    theme_classic,
)

mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["svg.fonttype"] = "none"
mpl.rcParams["font.family"] = "DejaVu Sans"


In [ ]:
BASE = Path(
    "/oak/stanford/groups/engreitz/Users/tri/10xPerturbSeq/"
    "analysis/250507_ePAH1_NovaSeq/CRISPR_count/"
    "IP_GV_unified_sceptre_cnmf_no_celltype"
)

RUN_NAME = "IP_GV_skcnmf_K30_dt2_0_n100"
RUN_ROOT = BASE / "PerturbNMF" / "results" / RUN_NAME

SPECTRA_PATH = (
    RUN_ROOT
    / "Inference"
    / "Inference.gene_spectra_score.k_30.dt_2_0.txt"
)

OUTPUT_DIR = (
    RUN_ROOT
    / "Evaluation"
    / "30_2_0"
    / "selected_program_umap"
    / "marker_rank_heatmap"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MAX_RANK_TO_SHOW = 300

PROGRAMS = [
    {
        "program": "P15",
        "number": 15,
        "label": "Homeostatic EC",
        "markers": ["Sfrp1", "Fbln2", "Cdh13", "Efna5"],
    },
    {
        "program": "P9",
        "number": 9,
        "label": "Angiogenic migration",
        "markers": ["Olfm2", "Atp13a3", "Cxcl12", "Pde1c"],
    },
    {
        "program": "P6",
        "number": 6,
        "label": "Inflammatory adhesion",
        "markers": ["Prss23", "Vwf", "Selp", "Ackr3"],
    },
    {
        "program": "P12",
        "number": 12,
        "label": "sGC-cGMP remodeling",
        "markers": ["Gucy1b1", "Gucy1a1", "Postn", "Pde5a"],
    },
]

print("Spectra:", SPECTRA_PATH)
print("Output:", OUTPUT_DIR)


In [ ]:
def parse_integer(value):
    match = re.search(r"(\d+)$", str(value))
    return int(match.group(1)) if match else None


def load_spectra(path: Path, expected_k: int = 30):
    if not path.exists():
        candidates = sorted(
            path.parent.glob(
                f"*gene_spectra_score*k_{expected_k}*dt_2_0*.txt"
            )
        )
        if len(candidates) != 1:
            raise FileNotFoundError(
                f"Could not find {path}; candidates: {candidates}"
            )
        path = candidates[0]

    spectra = pd.read_csv(path, sep="\t", index_col=0)

    if spectra.shape[0] == expected_k:
        oriented = spectra.copy()
    elif spectra.shape[1] == expected_k:
        oriented = spectra.transpose().copy()
    else:
        raise ValueError(
            f"Expected one spectra dimension to be {expected_k}; "
            f"observed {spectra.shape}."
        )

    parsed = [parse_integer(value) for value in oriented.index]
    if all(value is not None for value in parsed):
        if set(parsed) == set(range(expected_k)):
            program_numbers = [value + 1 for value in parsed]
        elif set(parsed) == set(range(1, expected_k + 1)):
            program_numbers = parsed
        else:
            program_numbers = list(range(1, expected_k + 1))
    else:
        program_numbers = list(range(1, expected_k + 1))

    oriented.index = pd.Index(
        program_numbers,
        name="program_number",
    )
    oriented.columns = oriented.columns.astype(str)
    oriented = oriented.apply(pd.to_numeric, errors="coerce")

    return oriented, path


spectra, resolved_spectra_path = load_spectra(SPECTRA_PATH)
print("Resolved spectra:", resolved_spectra_path)
print("Shape:", spectra.shape)
display(spectra.iloc[:4, :6])


In [ ]:
gene_order = [
    gene
    for program in PROGRAMS
    for gene in program["markers"]
]
program_order = [program["program"] for program in PROGRAMS]
row_labels = {
    program["program"]: (
        f'{program["program"]}\n{program["label"]}'
    )
    for program in PROGRAMS
}

column_lookup = {}
for column in spectra.columns:
    column_lookup.setdefault(str(column).upper(), str(column))

rows = []
missing_genes = []

for program in PROGRAMS:
    number = int(program["number"])
    if number not in spectra.index:
        raise KeyError(
            f"P{number} was not found in the spectra index."
        )

    scores = spectra.loc[number].dropna().sort_values(
        ascending=False
    )
    rank_lookup = pd.Series(
        np.arange(1, len(scores) + 1),
        index=scores.index.astype(str),
    )

    rank_lookup_upper = {
        str(gene).upper(): int(rank)
        for gene, rank in rank_lookup.items()
    }

    for gene in gene_order:
        gene_key = gene.upper()
        rank = rank_lookup_upper.get(gene_key)
        if rank is None:
            missing_genes.append(
                {"program": program["program"], "gene": gene}
            )

        shown = (
            rank is not None
            and rank <= MAX_RANK_TO_SHOW
        )
        rows.append(
            {
                "program": program["program"],
                "program_number": number,
                "program_label": row_labels[
                    program["program"]
                ],
                "gene": gene,
                "rank": rank,
                "rank_label": str(rank) if shown else "",
                "rank_score": (
                    1.0 / math.log2(rank + 1.0)
                    if shown
                    else 0.0
                ),
                "within_display_cutoff": shown,
            }
        )

plot_data = pd.DataFrame(rows)
plot_data["gene"] = pd.Categorical(
    plot_data["gene"],
    categories=gene_order,
    ordered=True,
)
plot_data["program_label"] = pd.Categorical(
    plot_data["program_label"],
    categories=list(reversed(
        [row_labels[x] for x in program_order]
    )),
    ordered=True,
)

plot_data.to_csv(
    OUTPUT_DIR
    / "IP_GV_prioritized_program_marker_ranks.csv",
    index=False,
)

if missing_genes:
    print("Missing gene/program combinations:")
    display(pd.DataFrame(missing_genes))

display(
    plot_data.loc[
        plot_data["within_display_cutoff"]
    ].sort_values(["program_number", "rank"])
)


In [ ]:
boundaries = [4.5, 8.5, 12.5]

plot = (
    ggplot(
        plot_data,
        aes(
            x="gene",
            y="program_label",
            fill="rank_score",
        ),
    )
    + geom_tile(color="#F0ECE9", size=0.45)
    + geom_text(
        aes(label="rank_label"),
        size=8.0,
        color="#111111",
    )
    + geom_vline(
        xintercept=boundaries,
        color="#C8C8C8",
        size=0.55,
    )
    + scale_fill_gradient(
        low="#FFF8F5",
        high="#7F0000",
        limits=(0, 1),
        name="1/log2(rank + 1)",
    )
    + labs(
        title="Prioritized program marker genes",
        subtitle=(
            f"Numbers show cNMF rank; ranks > "
            f"{MAX_RANK_TO_SHOW} are left blank"
        ),
        x="",
        y="",
    )
    + theme_classic()
    + theme(
        figure_size=(11.4, 3.8),
        plot_title=element_text(
            size=15,
            weight="bold",
            ha="center",
            margin={"b": 3},
        ),
        plot_subtitle=element_text(
            size=9,
            ha="center",
            color="#4D4D4D",
            margin={"b": 8},
        ),
        axis_text_x=element_text(
            rotation=52,
            ha="right",
            size=10,
            style="italic",
        ),
        axis_text_y=element_text(
            size=10.5,
            ha="right",
            linespacing=1.15,
        ),
        axis_title=element_blank(),
        axis_ticks=element_blank(),
        axis_line=element_blank(),
        panel_grid=element_blank(),
        legend_title=element_text(size=9),
        legend_text=element_text(size=8),
        legend_position="right",
        legend_key_height=75,
        legend_key_width=12,
        plot_margin=0.03,
    )
)

PDF_PATH = (
    OUTPUT_DIR
    / "IP_GV_prioritized_program_marker_rank_heatmap.pdf"
)
SVG_PATH = (
    OUTPUT_DIR
    / "IP_GV_prioritized_program_marker_rank_heatmap.svg"
)
PNG_PATH = (
    OUTPUT_DIR
    / "IP_GV_prioritized_program_marker_rank_heatmap.png"
)

plot.save(PDF_PATH, verbose=False)
plot.save(SVG_PATH, verbose=False)
plot.save(PNG_PATH, dpi=300, verbose=False)

display(plot)
print(PDF_PATH)
print(SVG_PATH)
print(PNG_PATH)
